- Load them with PyPDFLoader:, then split with RecursiveCharacterTextSplitter (chunk size 1000, overlap 200).
- Print how many pieces you got. That is the end of this session.

Part 1

In [9]:
!pip install -q langchain-community pypdf langchain-text-splitters

In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [11]:
# 1. Find every PDF that are uploaded
pdf_files = list(Path("/content").glob("*.pdf"))

In [12]:
# 2. Read all pages from all PDFs
documents = []
for pdf_file in pdf_files:
    loader = PyPDFLoader(str(pdf_file))
    documents.extend(loader.load())
print("Number of PDF files:", len(pdf_files))
print("Number of pages loaded:", len(documents))

Number of PDF files: 3
Number of pages loaded: 50


In [13]:
# 3. Cut the pages into small overlapping pieces
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print("Total chunks created:", len(chunks))

Total chunks created: 227


In [14]:
#  file names with page counts, and readable English text
from collections import Counter
print(Counter(Path(d.metadata["source"]).name for d in documents))
print(chunks[0].page_content[:300])

Counter({'RAG.pdf': 19, 'BERT.pdf': 16, 'AttentionIsAllYouNeed.pdf': 15})
BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout}@google.com
Abstract
We introduce a new language representa-
tion model called BERT, which stands f


Part 2

In [15]:
!pip install -q langchain-community langchain-huggingface sentence-transformers faiss-cpu pypdf

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, embeddings)
print("Index built")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index built


In [17]:
# Test one question first
results = db.similarity_search("What does BERT stand for?", k=3)
for r in results:
    print(r.metadata.get("source"), "page", r.metadata.get("page", 0) + 1)
    print(r.page_content[:300])
    print("---")

/content/BERT.pdf page 4
Input/Output Representations To make BERT
handle a variety of down-stream tasks, our input
representation is able to unambiguously represent
both a single sentence and a pair of sentences
(e.g.,⟨ Question, Answer⟩) in one token sequence.
Throughout this work, a “sentence” can be an arbi-
trary span 
---
/content/BERT.pdf page 3
ing pre-training, the model is trained on unlabeled
data over different pre-training tasks. For ﬁne-
tuning, the BERT model is ﬁrst initialized with
the pre-trained parameters, and all of the param-
eters are ﬁne-tuned using labeled data from the
downstream tasks. Each downstream task has sep-
arate
---
/content/BERT.pdf page 13
BERT (Ours)
Trm Trm Trm
Trm Trm Trm
...
...
Trm Trm Trm
Trm Trm Trm
...
...
OpenAI GPT
Lstm
ELMo
Lstm Lstm
Lstm Lstm Lstm
Lstm Lstm Lstm
Lstm Lstm Lstm
 T1 T2  TN...
...
...
...
...
 E1 E2  EN...
 T1 T2 TN...
 E1 E2  EN...
 T1 T2  TN...
 E1 E2  EN...
Figure 3: Differences in pre-training model archi
---


In [18]:
questions = [
    "What problem with recurrent models does the Transformer paper address?",
    "What is multi-head attention?",
    "How many layers does the base Transformer encoder have?",
    "What does BERT stand for?",
    "What two pre-training tasks does BERT use?",
    "How is BERT different from left-to-right language models?",
    "How does the RAG paper define RAG?",
    "What are the two RAG variants?",
    "Which retriever does the RAG model use?",
    "Which tasks was RAG evaluated on?",
]

for i, q in enumerate(questions, 1):
    print(f"Q{i}: {q}")
    for r in db.similarity_search(q, k=3):
        src = r.metadata.get("source", "").split("/")[-1]
        print("  -", src, "page", r.metadata.get("page", 0) + 1)
        print("    ", r.page_content[:200].replace(chr(10), " "))
    print()

Q1: What problem with recurrent models does the Transformer paper address?
  - BERT.pdf page 8
     We can see that larger models lead to a strict ac- curacy improvement across all four datasets, even for MRPC which only has 3,600 labeled train- ing examples, and is substantially different from the 
  - BERT.pdf page 12
     representation towards the actual observed word. The advantage of this procedure is that the Transformer encoder does not know which words it will be asked to predict or which have been re- placed by 
  - AttentionIsAllYouNeed.pdf page 2
     sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across examples. Recent work has achieved 

Q2: What is multi-head attention?
  - AttentionIsAllYouNeed.pdf page 3
     3.2 Attention An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and ou

In [ ]:
## Score - 3/10 Correct Answers